In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# Load cleaned train dataset
train_df = pd.read_csv('../data/Industrial Risk and Reliability Assessment/cleaned_train_data.csv')
print("Train Data Shape:", train_df.shape)

Train Data Shape: (61249, 27)


In [3]:
# Define feature columns
feature_cols = [col for col in train_df.columns if col not in ['engine_id', 'cycle', 'RUL']]

In [4]:
# Function to create sequences for LSTM (e.g., sequence length = 30)
def create_sequences(df, feature_cols, seq_length=30):
    sequences = []
    targets = []
    
    for engine_id, group in df.groupby('engine_id'):
        feature_values = group[feature_cols].values
        rul_values = group['RUL'].values
        
        if len(feature_values) >= seq_length:
            for i in range(len(feature_values) - seq_length):
                sequences.append(feature_values[i:(i + seq_length)])
                targets.append(rul_values[i + seq_length])
                
    return np.array(sequences), np.array(targets)

In [6]:
SEQ_LENGTH = 30
X_seq, y_seq = create_sequences(train_df, feature_cols, seq_length=SEQ_LENGTH)

print("X_seq shape:", X_seq.shape)
print("y_seq shape:", y_seq.shape)


X_seq shape: (53779, 30, 24)
y_seq shape: (53779,)


In [9]:
# Define LSTM Model
class RUL_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(RUL_LSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        # Initialize hidden and cell states with zeros
        h_0 = torch.zeros(2, x.size(0), 64).to(x.device)
        c_0 = torch.zeros(2, x.size(0), 64).to(x.device)
        
        # Forward pass through LSTM
        out, _ = self.lstm(x, (h_0, c_0))
        
        # Take the output from the last time step
        out = self.fc(out[:, -1, :])
        return out

In [10]:
# Initialize LSTM model
input_dim = len(feature_cols)
lstm_model = RUL_LSTM(input_dim=input_dim)
print(lstm_model)

RUL_LSTM(
  (lstm): LSTM(24, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [11]:
# Convert numpy arrays to PyTorch tensors
X_tensor = torch.tensor(X_seq, dtype=torch.float32)
y_tensor = torch.tensor(y_seq, dtype=torch.float32).unsqueeze(1)

In [12]:
# Create TensorDataset and DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [13]:
# Loss function and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

In [14]:
# Training Loop
epochs = 3
lstm_model.train()

for epoch in range(epochs):
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = lstm_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

print("LSTM Model Training Completed Successfully!")

Epoch [1/3], Loss: 15170.7402
Epoch [2/3], Loss: 9232.2379
Epoch [3/3], Loss: 7338.7521
LSTM Model Training Completed Successfully!


In [15]:
import os

# Save the trained LSTM model state dict
torch.save(lstm_model.state_dict(), '../saved_models/rul_lstm_model.pth')
print("LSTM Model saved successfully inside 'saved_models' folder!")

LSTM Model saved successfully inside 'saved_models' folder!
